# 🚀 Uni-MuMER Training + DagsHub Complete Upload

**Notebook này sẽ:**
1. Train model Uni-MuMER trên Kaggle GPU
2. Upload **TẤT CẢ** artifacts lên DagsHub/MLflow:
   - Training logs, metrics, configs
   - **TOÀN BỘ thư mục saves/** (~643MB checkpoint)
   - Evaluation results
3. Bạn có thể tải về từ DagsHub sau này để phân tích

## ⚙️ Kaggle Secrets Required:
- `DAGSHUB_TOKEN` - DagsHub access token
- `DAGSHUB_USERNAME` - Your DagsHub username
- `DAGSHUB_REPO_NAME` - Repo name (e.g., test-unimer)
- `HF_TOKEN` (optional) - Hugging Face token

## 📦 Step 1: Setup Environment & Clone Repo

In [ ]:
# Clean workspace
!rm -rf /kaggle/working/*
print("✅ Workspace cleaned")

In [ ]:
%%time
# Install Miniconda
!wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!bash Miniconda3-latest-Linux-x86_64.sh -b -f -p /kaggle/working/miniconda
!rm Miniconda3-latest-Linux-x86_64.sh

# Accept Anaconda terms
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main -q
!/kaggle/working/miniconda/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r -q

# Create environment
!/kaggle/working/miniconda/bin/conda create -n unimumer python=3.10 -y -q

print("✅ Miniconda installed")

In [ ]:
import os

# Setup activation command
ACTIVATE = "source /kaggle/working/miniconda/bin/activate unimumer"

# Clone repository
!git clone https://github.com/NhatPot/test-unimer.git
%cd test-unimer

print("✅ Repository cloned")

## 🔐 Step 2: Load Kaggle Secrets & Setup DagsHub

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

# Load secrets
secrets = UserSecretsClient()

try:
    # DagsHub credentials
    DAGSHUB_TOKEN = secrets.get_secret("DAGSHUB_TOKEN")
    DAGSHUB_USERNAME = secrets.get_secret("DAGSHUB_USERNAME")
    DAGSHUB_REPO_NAME = secrets.get_secret("DAGSHUB_REPO_NAME")
    
    # Set environment variables (DO NOT PRINT TOKENS)
    os.environ["DAGSHUB_TOKEN"] = DAGSHUB_TOKEN
    os.environ["DAGSHUB_USERNAME"] = DAGSHUB_USERNAME
    os.environ["DAGSHUB_REPO_NAME"] = DAGSHUB_REPO_NAME
    
    # MLflow tracking
    MLFLOW_TRACKING_URI = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO_NAME}.mlflow"
    os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    
    print("✅ DagsHub credentials loaded")
    print(f"   Username: {DAGSHUB_USERNAME}")
    print(f"   Repo: {DAGSHUB_REPO_NAME}")
    print(f"   MLflow: {MLFLOW_TRACKING_URI}")
    
except Exception as e:
    print(f"❌ ERROR: Could not load DagsHub secrets")
    print(f"   {e}")
    print("\n⚠️  Add these secrets to Kaggle:")
    print("   - DAGSHUB_TOKEN")
    print("   - DAGSHUB_USERNAME")
    print("   - DAGSHUB_REPO_NAME")
    raise

# Optional: Hugging Face token
try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("✅ HF token loaded")
except:
    print("ℹ️  No HF_TOKEN (optional)")

## 📥 Step 3: Download Base Model

In [ ]:
%%time
MODEL_DIR = "Uni-MuMER-Qwen2.5-VL-3B"

# Create model directory
!mkdir -p {MODEL_DIR}

# Download model shards
print("📥 Downloading base model (this may take a few minutes)...")

!wget -q --show-progress -L -O {MODEL_DIR}/model-00001-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00001-of-00002.safetensors?download=true"

!wget -q --show-progress -L -O {MODEL_DIR}/model-00002-of-00002.safetensors \
  "https://huggingface.co/phxember/Uni-MuMER-Qwen2.5-VL-3B/resolve/main/model-00002-of-00002.safetensors?download=true"

print("\n✅ Base model downloaded")
!ls -lh {MODEL_DIR}/*.safetensors

## 📚 Step 4: Install Dependencies

In [ ]:
%%time
# Install requirements (includes dagshub and mlflow)
!{ACTIVATE} && pip install -q -r requirements.txt

print("✅ Requirements installed")

In [ ]:
%%time
# Install LLaMA-Factory
%cd train/LLaMA-Factory
!{ACTIVATE} && pip install -q -e .
%cd ../..

print("✅ LLaMA-Factory installed")

## 🖥️ Step 5: Check GPU & Auto-Configure

In [ ]:
import subprocess
import yaml

# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

gpu_info = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True
)
gpu_name = gpu_info.stdout.strip()
os.environ["GPU_TYPE"] = gpu_name

print(f"\nGPU: {gpu_name}")

# Auto-configure for T4
is_t4 = "T4" in gpu_name

yaml_path = "train/Uni-MuMER-train.yaml"
with open(yaml_path, 'r') as f:
    config = yaml.safe_load(f)

if is_t4:
    print("⚠️  T4 detected → Using FP16 instead of BF16")
    config['bf16'] = False
    config['fp16'] = True
else:
    print("✅ Non-T4 GPU → BF16 may be available")

# Disable built-in reporting (we'll log manually)
config['report_to'] = 'none'

# Save config
with open(yaml_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print(f"✅ Config updated: fp16={config['fp16']}, bf16={config['bf16']}, report_to=none")

## 🔧 Step 6: Initialize DagsHub/MLflow Connection

In [ ]:
import dagshub
import mlflow

# Initialize DagsHub
try:
    dagshub.init(
        repo_owner=os.environ['DAGSHUB_USERNAME'],
        repo_name=os.environ['DAGSHUB_REPO_NAME'],
        mlflow=True
    )
    print("✅ DagsHub initialized")
    
    # Configure MLflow
    mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
    print(f"✅ MLflow tracking URI: {os.environ['MLFLOW_TRACKING_URI']}")
    
    # Test connection
    mlflow.set_experiment("Uni-MuMER-Qwen2.5VL")
    print("✅ MLflow connection verified")
    
except Exception as e:
    print(f"❌ DagsHub initialization failed: {e}")
    print("   Training will continue but without tracking")

## 🏋️ Step 7: Run Training

In [ ]:
%%time
# Run training
print("🚀 Starting training...\n")

!{ACTIVATE} && \
  MPLBACKEND=Agg llamafactory-cli train train/Uni-MuMER-train.yaml

print("\n✅ Training completed")

## 📊 Step 8: Upload EVERYTHING to DagsHub

In [ ]:
import yaml
from pathlib import Path
from datetime import datetime

# Load config
with open('train/Uni-MuMER-train.yaml', 'r') as f:
    config = yaml.safe_load(f)

output_dir = config['output_dir']
print(f"Output directory: {output_dir}")

# Find latest checkpoint
output_path = Path(output_dir)
checkpoints = sorted(output_path.glob("checkpoint-*"))

if checkpoints:
    latest_checkpoint = str(checkpoints[-1])
    print(f"Latest checkpoint: {latest_checkpoint}")
else:
    latest_checkpoint = output_dir
    print("⚠️  No checkpoint-* found, using output_dir")

# Check checkpoint size
!du -sh {latest_checkpoint}

# Generate run name
dataset_name = config.get('dataset', '')
if isinstance(dataset_name, list):
    dataset_str = ','.join(dataset_name) if len(dataset_name) <= 3 else f"{len(dataset_name)}_datasets"
    if 'error' in str(dataset_name).lower():
        task = "EDL"
    elif 'tree' in str(dataset_name).lower():
        task = "TreeCoT"
    elif '_can' in str(dataset_name).lower():
        task = "SymbolCount"
    else:
        task = "MultiTask"
else:
    task = "Training"
    dataset_str = str(dataset_name)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
epochs = config.get('num_train_epochs', 1)
rank = config.get('lora_rank', 64)
run_name = f"{task}_{epochs}ep_rank{rank}_{timestamp}"

print(f"\nRun name: {run_name}")
print(f"Task: {task}")
print(f"Datasets: {dataset_str}")

In [ ]:
%%time
# Upload everything to DagsHub via MLflow
print("📤 Uploading to DagsHub...\n")

!{ACTIVATE} && \
  python scripts/post_training_logger.py \
    --yaml-config train/Uni-MuMER-train.yaml \
    --output-dir {output_dir} \
    --checkpoint-dir {latest_checkpoint} \
    --run-name "{run_name}" \
    --dataset-info train/dataset_info.json

print("\n✅ Upload to DagsHub completed")

## 📦 Step 9: Create Kaggle Output Archive (Backup)

In [ ]:
# Verify important files exist
important_files = [
    f"{latest_checkpoint}/adapter_model.safetensors",
    f"{latest_checkpoint}/adapter_config.json",
    f"{output_dir}/trainer_state.json",
    f"{output_dir}/README_RUN.md",
    f"{output_dir}/ARTIFACT_MANIFEST.json",
]

print("📋 Verifying files:")
all_exist = True
for file_path in important_files:
    exists = os.path.exists(file_path)
    status = "✅" if exists else "❌"
    print(f"{status} {file_path}")
    if not exists:
        all_exist = False

if all_exist:
    print("\n✅ All critical files present")
else:
    print("\n⚠️  Some files missing")

In [ ]:
%%time
# Create backup archive
archive_name = f"unimumer_{task}_{timestamp}.tar.gz"
archive_path = f"/kaggle/working/{archive_name}"

print(f"📦 Creating backup archive: {archive_name}\n")

# Archive the entire checkpoint directory
!tar -czf {archive_path} \
  -C {output_dir} \
  . \
  2>&1 | head -20

# Check result
if os.path.exists(archive_path):
    size_mb = os.path.getsize(archive_path) / (1024 * 1024)
    print(f"\n✅ Archive created: {archive_path}")
    print(f"   Size: {size_mb:.2f} MB")
else:
    print(f"\n⚠️  Archive not created")

## 📊 Step 10: Summary & Access Info

In [ ]:
print("=" * 70)
print("🎉 TRAINING & UPLOAD HOÀN TẤT")
print("=" * 70)

print("\n📁 Local Files:")
print(f"   Output dir: {output_dir}")
print(f"   Checkpoint: {latest_checkpoint}")
print(f"   Archive: {archive_path}")

print("\n☁️  DagsHub Upload:")
print("   ✅ Training config & parameters")
print("   ✅ Training logs & metrics")
print("   ✅ Full checkpoint (~643MB)")
print("   ✅ README_RUN.md")
print("   ✅ ARTIFACT_MANIFEST.json")

dagshub_url = f"https://dagshub.com/{os.environ.get('DAGSHUB_USERNAME')}/{os.environ.get('DAGSHUB_REPO_NAME')}"
mlflow_url = f"{dagshub_url}.mlflow"

print("\n🌐 Xem kết quả tại:")
print(f"   DagsHub: {dagshub_url}")
print(f"   MLflow: {mlflow_url}")
print(f"\n   → Click 'Experiments' tab để xem run: {run_name}")

print("\n📥 Tải về từ DagsHub:")
print("   1. Vào MLflow UI")
print(f"   2. Tìm run: {run_name}")
print("   3. Tab 'Artifacts' → Download checkpoint")
print("   4. Hoặc dùng MLflow API:")
print(f"      mlflow artifacts download -r <run-id> -d ./download")

print("\n💾 Kaggle Output:")
print(f"   {archive_name} (backup)")

print("\n" + "=" * 70)
print("✅ Done! All data uploaded to DagsHub for analysis.")
print("=" * 70)

## 📖 How to Download from DagsHub

### Method 1: Web UI
1. Go to your DagsHub repo
2. Click **Experiments** tab
3. Find your run by name
4. Click **Artifacts** tab
5. Download individual files or entire checkpoint folder

### Method 2: MLflow CLI
```bash
# Set credentials
export MLFLOW_TRACKING_URI=https://dagshub.com/username/repo.mlflow
export MLFLOW_TRACKING_USERNAME=your-username
export MLFLOW_TRACKING_PASSWORD=your-token

# Download artifacts
mlflow artifacts download -r <run-id> -d ./artifacts
```

### Method 3: Python API
```python
import mlflow
import os

os.environ['MLFLOW_TRACKING_URI'] = 'https://dagshub.com/username/repo.mlflow'
os.environ['MLFLOW_TRACKING_USERNAME'] = 'your-username'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your-token'

# Download artifacts
mlflow.artifacts.download_artifacts(
    run_id='your-run-id',
    artifact_path='checkpoint',
    dst_path='./downloaded'
)
```

## 📊 What You Can Analyze

From DagsHub artifacts:
- **Training curves**: Loss, learning rate over time
- **Hyperparameters**: All training config
- **Checkpoint**: Load adapter for inference or continue training
- **Logs**: Detailed training logs
- **Metrics**: Final training metrics
- **Manifest**: SHA256 checksums of all files

## 🔄 Continue Training

```python
# Download checkpoint from DagsHub
# Then in your YAML:
resume_from_checkpoint: path/to/downloaded/checkpoint
```